In [ ]:
!pip install -q "sagemaker<3"
%pip install -q scikit-learn pandas

In [ ]:
import sagemaker, boto3, sklearn
print("sagemaker:", sagemaker.__version__)
print("region   :", boto3.Session().region_name)

In [ ]:
import sagemaker, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

session = sagemaker.Session()
bucket  = "ty-aids-sagemaker-7-12414219"
prefix  = "breast-cancer"
role    = "arn:aws:iam::127774031942:role/7_SageMakerAssignmentRole"

data = load_breast_cancer(as_frame=True)
df = data.frame                          # 30 features + 'target'
print("Shape:", df.shape)
print(df["target"].value_counts().rename({0: "malignant", 1: "benign"}))

train, test = train_test_split(df, test_size=0.2, stratify=df["target"],
                               random_state=42)
print("Train:", train.shape, " Test:", test.shape)

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

train_s3 = session.upload_data("train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
test_s3  = session.upload_data("test.csv",  bucket=bucket, key_prefix=f"{prefix}/test")
print(train_s3)
print(test_s3)

In [ ]:
%%writefile train.py
import argparse, os, joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

def model_fn(model_dir):                       # used by the inference container
    return joblib.load(os.path.join(model_dir, "model.joblib"))

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--n-estimators", type=int, default=200)
    p.add_argument("--max-depth", type=int, default=8)
    p.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR"))
    p.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN"))
    p.add_argument("--test", default=os.environ.get("SM_CHANNEL_TEST"))
    args = p.parse_args()

    tr = pd.read_csv(os.path.join(args.train, "train.csv"))
    te = pd.read_csv(os.path.join(args.test, "test.csv"))
    X_tr, y_tr = tr.drop("target", axis=1), tr["target"]
    X_te, y_te = te.drop("target", axis=1), te["target"]

    model = RandomForestClassifier(n_estimators=args.n_estimators,
                                   max_depth=args.max_depth, random_state=42)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()

    print(f"test_accuracy={accuracy_score(y_te, pred):.4f}")
    print(f"test_precision={precision_score(y_te, pred):.4f}")
    print(f"test_recall={recall_score(y_te, pred):.4f}")
    print(f"test_f1={f1_score(y_te, pred):.4f}")
    print(f"test_auc={roc_auc_score(y_te, prob):.4f}")
    print(f"test_recall_malignant={recall_score(y_te, pred, pos_label=0):.4f}")
    print(f"confusion_tn={tn} confusion_fp={fp} confusion_fn={fn} confusion_tp={tp}")

    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))

In [ ]:
!mkdir -p /tmp/m && SM_MODEL_DIR=/tmp/m SM_CHANNEL_TRAIN=. SM_CHANNEL_TEST=. python train.py

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

metrics = [
    {"Name": "test:accuracy",         "Regex": "test_accuracy=([0-9\\.]+)"},
    {"Name": "test:precision",        "Regex": "test_precision=([0-9\\.]+)"},
    {"Name": "test:recall",           "Regex": "test_recall=([0-9\\.]+)"},
    {"Name": "test:f1",               "Regex": "test_f1=([0-9\\.]+)"},
    {"Name": "test:auc",              "Regex": "test_auc=([0-9\\.]+)"},
    {"Name": "test:recall_malignant", "Regex": "test_recall_malignant=([0-9\\.]+)"},
]

est1 = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={"n-estimators": 200, "max-depth": 8},
    metric_definitions=metrics,
    output_path=f"s3://{bucket}/{prefix}/output",
    base_job_name="bc-rf-run1-d8-n200",
)
est1.fit({"train": train_s3, "test": test_s3})
print("Model artifact:", est1.model_data)

In [ ]:
est2 = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={"n-estimators": 100, "max-depth": 4},
    metric_definitions=metrics,
    output_path=f"s3://{bucket}/{prefix}/output",
    base_job_name="bc-rf-run2-d4-n100",
)
est2.fit({"train": train_s3, "test": test_s3})
print("Model artifact:", est2.model_data)

In [ ]:
import pandas as pd
results = pd.DataFrame({
    "Run 1 (depth 8, 200 trees)": [0.9561, 0.9589, 0.9722, 0.9655, 0.9940, 0.9286, 39, 3, 2, 70, 124, 124],
    "Run 2 (depth 4, 100 trees)": [0.9561, 0.9589, 0.9722, 0.9655, 0.9950, 0.9286, 39, 3, 2, 70, 134, 134],
}, index=["Accuracy", "Precision", "Recall (benign)", "F1", "ROC-AUC", "Recall (malignant)",
          "TN", "FP (false benign)", "FN", "TP", "Training seconds", "Billable seconds"])
results

In [ ]:
predictor = est2.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="breast-cancer-rf-endpoint",
)

In [ ]:
sample = test.drop("target", axis=1).iloc[:5].values
print("Predicted:", predictor.predict(sample))
print("Actual   :", test["target"].iloc[:5].values)

In [ ]:
%pip install -q matplotlib

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

X_test = test.drop("target", axis=1).values
y_true = test["target"].values
y_pred = predictor.predict(X_test)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print(cm)
print(classification_report(y_true, y_pred,
      target_names=["malignant", "benign"], digits=4))

disp = ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - live endpoint (114 test samples)")
plt.show()

In [ ]:
import json
benign    = test[test.target == 1].drop("target", axis=1).iloc[0].tolist()
malignant = test[test.target == 0].drop("target", axis=1).iloc[0].tolist()

json.dump({"features": benign},    open("benign.json", "w"))
json.dump({"features": malignant}, open("malignant.json", "w"))

print(json.dumps({"body": json.dumps({"features": malignant})}))